In [1]:
import gymnasium as gym
import numpy as np
import os
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output
import pandas as pd

In [2]:
env = gym.make("MountainCar-v0", render_mode="rgb_array", max_episode_steps=-1)
env.reset()

(array([-0.5500034,  0.       ], dtype=float32), {})

In [3]:
gamma = 0.99
epsilon = 0.9

max_epsilon = 0.9
min_epsilon = 0.0

decay = 0.003

learning_rate = 0.2

total_episodes = 15000
max_steps = 200

In [4]:
bins_position_amount = 50
bins_velocity_amount = 25

max_position = env.observation_space.high[0]
min_position = env.observation_space.low[0]
max_velocity = env.observation_space.high[1]
min_velocity = env.observation_space.low[1]

bins_position = np.linspace(min_position, max_position, bins_position_amount)
bins_velocity = np.linspace(min_velocity, max_velocity, bins_velocity_amount)

In [5]:
def calculate_value(Q, state, action, reward, next_state, alpha, gamma):
    Q[state[0], state[1], action] = Q[state[0], state[1], action] + alpha * (reward + gamma * np.max(Q[next_state[0], next_state[1]]) - Q[state[0], state[1], action])
    return Q

In [6]:
def get_state(observation):
    position = np.digitize(observation[0], bins=bins_position) - 1
    velocity = np.digitize(observation[1], bins=bins_velocity) - 1
    return position, velocity

In [7]:
for episode in range(total_episodes):
    env.reset()
    clear_output(wait=True)
    plt.imshow(env.render())
    plt.show()

    observation_old, info = env.reset()
    observation_old = get_state(observation_old)
    q_table = np.zeros((bins_position_amount, bins_velocity_amount, env.action_space.n))
    done = False
    current_rewards = 0

    for step in range(max_steps):
        if np.random.uniform(0, 1) > epsilon:
            action = np.argmax(q_table[observation_old[0], observation_old[1]])
        else:
            action = env.action_space.sample()

        observation, reward, done, info, truncate = env.step(action)
        observation = get_state(observation)

        if done and observation[0] >= 0.5:
            print(f"Steps taken: {step}")
            q_table = calculate_value(q_table, observation_old, action, 200, observation, learning_rate, gamma)
            current_rewards = current_rewards + reward
            print(f"Episode {episode} done, steps needed {step}, rewards: {current_rewards}, epsilon: {epsilon}")
            break

        q_table = calculate_value(q_table, observation_old, action, reward, observation, learning_rate, gamma)

        observation_old = observation
        current_rewards = current_rewards + reward

        """clear_output(wait=True)
        plt.imshow(env.render())
        plt.show()
        print(f'Step: {step}')
        print(f'Position: {observation[0]}')
        print(f'Velocity: {observation[1]}')

        # slow down the visual update
        time.sleep(0.01)"""

    epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay * episode)
env.close()

KeyboardInterrupt: 